In [2]:
import re
import os
from datetime import datetime
from pathlib import Path

In [9]:
def convert_madx_to_cosy(input_filename, output_filename):
    if not os.path.exists(input_filename):
        return f"Ошибка: Файл не найден: {input_filename}"

    with open(input_filename, 'r', encoding='utf-8') as f:
        data = f.read()

    # 1. Парсинг определений элементов
    elements = {}
    elem_pattern = re.compile(r'(\w+):\s*(\w+),\s*([^;]+);')
    for match in elem_pattern.finditer(data):
        name, etype, params_str = match.groups()
        p = {k.strip().upper(): v.strip() for k, v in 
             (pair.split('=') for pair in params_str.split(',') if '=' in pair)}
        elements[name.upper()] = {'type': etype.upper(), 'params': p}

    # 2. Парсинг LINE
    line_match = re.search(r'MACHINE:\s*LINE\s*=\s*\((.*?)\)', data, re.DOTALL)
    if not line_match: return "Ошибка: LINE не найдена."
    
    raw_line_text = line_match.group(1).replace('\n', '')
    sequence = [e.strip().upper() for e in raw_line_text.split(',')]

    cosy = []
    
    # Блок метаданных
    cosy.append("{ " + "*"*60 + " }")
    cosy.append("{ COSY Infinity Lattice Converter }")
    cosy.append("{ Purpose: Conversion from MADX to COSY Infinity }")
    cosy.append(f"{{ Author: Kolokolchikov S. }}")
    cosy.append(f"{{ Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} }}")
    cosy.append(f"{{ Input file: {input_filename:<48} }}")
    cosy.append(f"{{ Output file: {output_filename:<47} }}")
    cosy.append(f"{{ TOTAL ELEMENTS: {len(sequence)} }}")
    cosy.append("{ " + "*"*60 + " }\n")

    cosy.append("INCLUDE 'header';\n")
    cosy.append("PROCEDURE LATTICE SEXTGx1 SEXTGy1 SEXTGx2 SEXTGy2 EB1 RFFLAG;")
    
    # 4. Объявление переменных
    cosy.append(" VARIABLE VRF 1 1 1; VARIABLE FREQ 1; VARIABLE HNUM 1; VARIABLE I 1;")
    cosy.append(" VARIABLE A 1; VARIABLE EBE 1;")
    
    # Квадруполи, длины и углы
    for name, info in elements.items():
        if info['type'] == 'QUADRUPOLE':
            cosy.append(f" VARIABLE {name} 1;")
        cosy.append(f" VARIABLE L_{name} 1;")
        if info['type'] == 'SBEND':
            cosy.append(f" VARIABLE ANG_{name} 1;")

    cosy.append(" VARIABLE SF1 1; VARIABLE SF2 1; VARIABLE SD 1; VARIABLE SD1 1;")
    
    # 5. Инициализация параметров
    cosy.append("\n {LATTICE PARAMETERS}")
    cosy.append(" EBE := EB1; A := 0.05;")
    cosy.append(" SF1 := SEXTGx1; SF2 := SEXTGx2; SD := SEXTGy1; SD1 := SEXTGy2;")
    
    cosy.append("\n {ELEMENT VALUES FROM MADX}")
    for name, info in elements.items():
        cosy.append(f" L_{name} := {info['params'].get('L', '0')};")
        if info['type'] == 'QUADRUPOLE':
            cosy.append(f" {name} := {info['params'].get('K1', '0')};")
        if info['type'] == 'SBEND':
            cosy.append(f" ANG_{name} := {info['params'].get('ANGLE', '0')};")

    # 6. Блок ВЧ (RF)
    cosy.append("\n {SETTING RF PARAMETERS}")
    cosy.append(" HNUM := 66; VRF(1, 1) := 100/HNUM; FREQ := HNUM*REVFREQ(fACCLEN(1));")
    cosy.append(" UM; CR; IF RFFLAG=1; RF VRF 0 FREQ 0 0.05; ENDIF;")
    
    # 7. Структура LINE
    cosy.append("\n {BEGIN LATTICE}")
    for name in sequence:
        el = elements.get(name)
        if not el: continue
        if el['type'] == 'SBEND':
            cosy.append(f" SBEND L_{name} ANG_{name} 0 0 0 0 0 0.5 0.5 ; {{{name}}}")
        elif el['type'] == 'QUADRUPOLE':
            cosy.append(f" QUAD L_{name} A {name} ; {{{name}}}")
        elif el['type'] == 'DRIFT':
            cosy.append(f" DL L_{name} ; {{{name}}}")

    cosy.append("\nENDPROCEDURE;")
    
    cosy.append(f"\n{{elem-counting: {len(sequence)}}}")
    cosy.append(f"SAVE '{Path(output_filename).stem}';")

    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(cosy))
    
    return f"Готово. Обработано {len(sequence)} элементов."

In [10]:
# --- ПУТИ ---
input_file = "C:/GIT_REPOS/SCT/MADX/1.Sequences/QFS_deutron.seq"
output_file = "C:/GIT_REPOS/SCT/COSY/src/FS_deutron.fox"

print(convert_madx_to_cosy(input_file, output_file))

Готово. Обработано 53 элементов.
